# Constants, packages and path

In [30]:
# constants for LPG supply chain processing
default_LPG_price = 0.6  # USD/kg
STS_cost = 0.1  # 10%
pre_bottling_cost = 0.2  # 20% TO BE MODIFIED
CRUDE_BARREL_TO_TONS = 7.33  # barrel of crude per ton
BUFFER_KM = 10  # km radius for spatial buffer queries

import pandas as pd
import geopandas as gpd
from pathlib import Path
import numpy as np
import rasterio
from rasterio.mask import mask as raster_mask
from shapely.geometry import Point, box

# Set up data path
data_dir = Path("dataset_first_step")
data_dir.mkdir(exist_ok=True)

# create mask layer to use as base (just to try)

In [31]:
# Build mask_layer from friction raster footprint
from rasterio.features import shapes
from shapely.geometry import shape
from shapely.ops import unary_union

friction_mask_path = data_dir / "friction_moto.tif"
if not friction_mask_path.exists():
    raise FileNotFoundError(f"Missing friction raster for mask creation: {friction_mask_path}")

with rasterio.open(friction_mask_path) as src:
    if src.crs is None:
        raise ValueError("friction_moto.tif has no CRS")

    # Valid data footprint: non-nodata pixels
    valid_mask = src.dataset_mask() > 0
    if not np.any(valid_mask):
        raise ValueError("friction_moto.tif contains no valid pixels to build mask_layer")

    mask_shapes = [
        shape(geom)
        for geom, value in shapes(valid_mask.astype(np.uint8), mask=valid_mask, transform=src.transform)
        if value == 1
    ]

    if len(mask_shapes) == 0:
        raise ValueError("Could not extract a valid geometry footprint from friction_moto.tif")

    mask_geom = unary_union(mask_shapes)
    mask_layer = gpd.GeoDataFrame(geometry=[mask_geom], crs=src.crs).to_crs("EPSG:4326")

print(f"Mask layer initialized from friction raster ({len(mask_layer)} geometry)")
print("Constants declared and imports completed")

Mask layer initialized from friction raster (1 geometry)
Constants declared and imports completed


# Create useful function for data loading and fill missing values

In [33]:
# Load input data files with smart defaults
from onstove.layer import VectorLayer, RasterLayer

# Define default gpkg paths and load defaults once
DEFAULT_PATHS = {
    'refineries': Path("dataset_first_step") / "default_refineries.gpkg",
    'ports': Path("dataset_first_step") / "default_ports.gpkg",
    'gas_plants': Path("dataset_first_step") / "default_gas_plants.gpkg",
    'primary_storage': Path("dataset_first_step") / "default_primary_storage.gpkg",
}


def _as_vector_layer(gdf, name="mask"):
    vect = VectorLayer(name=name)
    vect.data = gdf.copy()
    return vect


def load_raster_means(raster_path, band_names, mask_gdf):
    """
    Read a multiband raster, clip by mask_gdf, and return band averages.

    Output format matches previous table usage:
    - index: band names
    - single numeric column
    """
    if not raster_path.exists():
        raise FileNotFoundError(f"Missing raster file: {raster_path}")

    if mask_gdf.crs is None:
        raise ValueError("mask_layer has no CRS")

    mask_vect = _as_vector_layer(mask_gdf, name="mask_layer")
    values = {}

    for band_idx, band_name in enumerate(band_names, start=1):
        raster = RasterLayer(name=f"{raster_path.stem}_{band_name}", path=str(raster_path), band=band_idx)
        raster.mask(mask_vect, crop=True, all_touched=False)

        arr = raster.data.astype(np.float64)
        nodata = raster.meta.get('nodata')
        if nodata is not None:
            try:
                if not np.isnan(nodata):
                    arr[arr == nodata] = np.nan
            except TypeError:
                arr[arr == nodata] = np.nan

        mean_val = np.nanmean(arr)
        if np.isnan(mean_val):
            raise ValueError(
                f"Band '{band_name}' in {raster_path.name} has no valid data inside mask_layer"
            )
        values[band_name] = float(mean_val)

    result = pd.DataFrame.from_dict(values, orient='index')
    result.columns = [1]
    return result


def load_or_use_default(gpkg_name, default_path, mask_gdf, data_dir=data_dir):
    """
    SMART DATA LOADING with fallback to defaults

    Two-phase strategy:
    1. If user provided data (file exists in dataset_first_step/), load it directly
    2. If not, load default file, clip it with mask_gdf, and keep it in memory
    """
    gpkg_path = data_dir / f"{gpkg_name}.gpkg"
    mask_vect = _as_vector_layer(mask_gdf, name="mask_layer")

    if gpkg_path.exists():
        # PHASE 1: User data found
        layer = VectorLayer(name=gpkg_name, path=str(gpkg_path))
        layer.mask(mask_vect, keep_geom_type=False)
        gdf = layer.data
        print(f"{gpkg_name}: loaded and clipped in memory ({len(gdf)} records)")
        return gdf

    # PHASE 2: Use default and clip with mask_layer (no file creation)
    default_layer = VectorLayer(name=gpkg_name, path=str(default_path))
    default_layer.mask(mask_vect, keep_geom_type=False)

    gdf = default_layer.data
    print(f"{gpkg_name}: clipped from default in memory ({len(gdf)} records)")
    return gdf


def fill_from_buffer(gdf, default_gdf, cols, buffer_km=BUFFER_KM):
    """
    FILL MISSING VALUES using spatial buffer queries
    
    For each geometry in gdf with NaN values in 'cols', search for the nearest value
    in default_gdf within a radius of buffer_km. Useful for:
    - Filling missing LPG prices from default data
    - Completing compliance fields (VLGC)
    - Filling production capacity from nearby facilities
    
    INPUT:
    - gdf (GeoDataFrame): main data with missing values
    - default_gdf (GeoDataFrame): default data for spatial fallback
    - cols (list): columns to fill, e.g., ['LPG_price', 'LPG_prod']
    - buffer_km (float): search radius in km (default: BUFFER_KM=10)
    
    LOGIC:
    1. Convert default_gdf to same CRS as gdf
    2. Calculate buffer_dist in degrees (lat/lon) or meters (projected)
    3. For each column and each row with NaN:
       - Create buffer around the geometry
       - Find default features intersecting the buffer
       - Take first non-null value from default_gdf
       - Assign it to gdf
    
    OUTPUT:
    - GeoDataFrame: gdf with NaN filled from nearest values in default_gdf
    
    EXAMPLE:
    refineries = fill_from_buffer(refineries, default_refineries, ['LPG_price', 'LPG_prod'])
    # If refineries has missing prices, searches in default_refineries within 10km
    """
    gdf, default_gdf = gdf.copy(), default_gdf.to_crs(gdf.crs)
    # Unit conversion: lat/lon uses degrees (~111 km per degree), projected uses meters
    buffer_dist = buffer_km / 111.0 if gdf.crs.is_geographic else buffer_km * 1000
    
    for col in cols:
        # Skip if column doesn't exist in either dataset
        if col not in gdf.columns:
            continue
        # Find rows with missing values (NaN or empty strings)
        missing_idx = gdf[gdf[col].isna() | (gdf[col] == '')].index
        if len(missing_idx) == 0:
            continue
        print(f"  Filling '{col}': {len(missing_idx)} rows")
        
        # For each row with missing value
        for idx in missing_idx:
            # Create buffer around the geometry
            port_buffer = gdf.loc[idx, 'geometry'].buffer(buffer_dist)
            # Find all default features inside the buffer
            nearby = default_gdf[default_gdf.geometry.intersects(port_buffer)]
            
            # If found at least one nearby feature with the value
            if len(nearby) > 0 and col in nearby.columns:
                val = nearby[col].dropna()
                if len(val) > 0:
                    # Assign the first value found
                    gdf.at[idx, col] = val.iloc[0]
    return gdf

# Create fictional default gas plant layer (just to try...)

In [34]:
# Create sample gas_plants if missing
if not (data_dir / "default_gas_plants.gpkg").exists():
    sample_data = {
        'geometry': [Point(3.8667, 6.4667), Point(5.1447, 5.9320), Point(6.1256, 4.9521)],
        'name': ['Gas Plant Lagos', 'Gas Plant Port Harcourt', 'Gas Plant Warri'],
        'LPG_prod': [150.0, 200.0, 120.0],
        'LPG_price': [0.75, 0.72, 0.78],
    }
    gpd.GeoDataFrame(sample_data, crs="EPSG:4326").to_file(data_dir / "default_gas_plants.gpkg", driver="GPKG")
    print("Sample default_gas_plants.gpkg created")

# data loading or default data use

In [35]:
# Load all data
try:
    refineries = load_or_use_default("refineries", DEFAULT_PATHS['refineries'], mask_layer)
    ports = load_or_use_default("ports", DEFAULT_PATHS['ports'], mask_layer)
    gas_plants = load_or_use_default("gas_plants", DEFAULT_PATHS['gas_plants'], mask_layer)
    primary_storage = load_or_use_default("primary_storage", DEFAULT_PATHS['primary_storage'], mask_layer)

    # Read multiband rasters and use average over mask_layer as actual values
    national_shares = load_raster_means(
        data_dir / "national_shares.tif",
        ['ports', 'refineries', 'gas_plants'],
        mask_layer,
    )
    lpg_import_cost = load_raster_means(
        data_dir / "lpg_import_cost.tif",
        ['import_land', 'import_sea'],
        mask_layer,
    )

    print("\nNational shares (mean over masked area):")
    print(national_shares)
    print("\nLPG import costs (mean over masked area):")
    print(lpg_import_cost)

    # Load defaults once (reuse for all fillings)
    default_data = {name: gpd.read_file(path) for name, path in DEFAULT_PATHS.items()}

    print("\n✓ Data loaded successfully")
except (FileNotFoundError, ValueError) as e:
    print(f"Data loading error: {e}")

refineries: clipped from default in memory (4 records)
ports: clipped from default in memory (11 records)
gas_plants: clipped from default in memory (3 records)
primary_storage: loaded and clipped in memory (19 records)


d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_ports layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_refineries layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_gas_plants layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The lpg_import_cost_import_land layer do not have 


National shares (mean over masked area):
                    1
ports       50.738055
refineries   0.063389
gas_plants  49.198556

LPG import costs (mean over masked area):
                    1
import_land  0.094122
import_sea   0.019422

✓ Data loaded successfully


# fill missing data from buffer or default + calculate port cost

In [50]:
# Process refineries: fill missing values and ensure key fields
def process_refineries(gdf, default_gdf):
    """
    REFINERIES PROCESSING
    
    Fill missing values from buffer and ensure key fields with defaults.
    
    INPUT:
    - gdf (GeoDataFrame): refineries data to process
    - default_gdf (GeoDataFrame): default refineries data for fallback values
    
    OUTPUT:
    - GeoDataFrame: refineries with LPG_prod and LPG_price filled and ensured
    
    EXAMPLE:
    refineries = process_refineries(refineries, default_data['refineries'])
    """
    gdf = gdf.copy()
    
    # Phase 1: Fill from buffer
    fill_cols = ['LPG_prod', 'LPG_price']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)
    
    # Phase 2: Ensure key fields and apply defaults
    if 'LPG_prod' not in gdf.columns:
        gdf['LPG_prod'] = 0.0
    else:
        gdf['LPG_prod'] = gdf['LPG_prod'].fillna(0.0)
    
    if 'LPG_price' not in gdf.columns:
        gdf['LPG_price'] = default_LPG_price
    else:
        gdf['LPG_price'] = gdf['LPG_price'].fillna(default_LPG_price)
    
    return gdf

def process_ports(gdf, default_gdf, storage_gdf, import_cost_df):
    """
    PORTS PROCESSING
    
    Fill missing values, find nearest storage facility, and adjust pricing based on compliance.
    
    INPUT:
    - gdf (GeoDataFrame): ports data to process
    - default_gdf (GeoDataFrame): default ports data for fallback values
    - storage_gdf (GeoDataFrame): storage/tank locations for finding nearest facility
    - import_cost_df (DataFrame): import cost multipliers for pricing adjustments
    
    PORT-SPECIFIC LOGIC:
    - LPG_capacity: taken from nearest primary_storage within BUFFER_KM
    - tanks_nearby: count/sum of storage facilities nearby
    - LPG_compliance: True if storage facilities found nearby, False otherwise
    - LPG_price differentiated by compliance:
      * With compliance (storage nearby):
        - VLGC ports: LPG_price × (1+import_sea)
        - Non-VLGC ports: LPG_price × (1+import_sea) × (1 + STS_cost)
      * Without compliance (no storage nearby):
        - All ports: LPG_price × (1+import_sea) × (1+pre_bottling_cost)
    
    OUTPUT:
    - GeoDataFrame: ports with LPG_capacity from storage, compliance, and adjusted pricing
    
    EXAMPLE:
    ports = process_ports(ports, default_data['ports'], primary_storage, lpg_import_cost)
    """
    gdf = gdf.copy()
    
    # Phase 1: Fill missing values from buffer
    fill_cols = ['VLGC_compliance', 'LPG_price']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)
    
    # Phase 2: Ensure LPG_price has defaults
    if 'LPG_price' not in gdf.columns:
        gdf['LPG_price'] = default_LPG_price
    else:
        gdf['LPG_price'] = gdf['LPG_price'].fillna(default_LPG_price)
    
    # Phase 3: Initialize port-specific fields
    gdf['LPG_capacity'] = 0.0
    gdf['tanks_nearby'] = 0.0
    gdf['LPG_compliance'] = False
    
    # Ensure VLGC_compliance exists and fill NaN with False
    if 'VLGC_compliance' not in gdf.columns:
        gdf['VLGC_compliance'] = False
    else:
        gdf['VLGC_compliance'] = gdf['VLGC_compliance'].fillna(False)
    
    gdf = gdf.to_crs(storage_gdf.crs) if storage_gdf is not None else gdf
    
    # Phase 4: Find nearest storage facility and get LPG_capacity
    if storage_gdf is not None:
        buffer_dist = BUFFER_KM / 111.0 if gdf.crs.is_geographic else BUFFER_KM * 1000
        for idx, port_geom in gdf.geometry.items():
            # Create buffer around port
            port_buffer = port_geom.buffer(buffer_dist)
            
            # Find all storage facilities within buffer
            nearby = storage_gdf[storage_gdf.geometry.intersects(port_buffer)]
            
            if len(nearby) > 0:
                # Sum of LPG_capacity from all nearby storage
                tanks_cap = nearby['LPG_capacity'].sum() if 'LPG_capacity' in nearby.columns else 0.0
                gdf.at[idx, 'LPG_capacity'] = tanks_cap
                gdf.at[idx, 'tanks_nearby'] = len(nearby)
                gdf.at[idx, 'LPG_compliance'] = True
            else:
                gdf.at[idx, 'LPG_capacity'] = 0.0
                gdf.at[idx, 'tanks_nearby'] = 0
                gdf.at[idx, 'LPG_compliance'] = False
    
    # Phase 5: Price adjustment based on LPG_compliance
    import_sea = import_cost_df.loc['import_sea'].iloc[0]
    
    # Ports WITH storage nearby (LPG_compliance = True)
    compliant = gdf['LPG_compliance'] == True
    gdf.loc[compliant, 'LPG_price'] = gdf.loc[compliant, 'LPG_price'] * (1+import_sea)
    # Add STS_cost for non-VLGC compliant ports
    non_vlgc_compliant = compliant & (gdf['VLGC_compliance'] != True)
    gdf.loc[non_vlgc_compliant, 'LPG_price'] *= (1 + STS_cost)
    
    # Ports WITHOUT storage nearby (LPG_compliance = False)
    non_compliant = gdf['LPG_compliance'] == False
    gdf.loc[non_compliant, 'LPG_price'] = gdf.loc[non_compliant, 'LPG_price'] * (1+import_sea) * (1+pre_bottling_cost)
    
    return gdf

def process_primary_storage(gdf, default_gdf):
    """
    PRIMARY STORAGE PROCESSING
    
    Fill missing values from buffer and ensure key fields with defaults.
    
    INPUT:
    - gdf (GeoDataFrame): primary storage data to process
    - default_gdf (GeoDataFrame): default storage data for fallback values
    
    OUTPUT:
    - GeoDataFrame: storage with LPG_capacity and LPG_price filled and ensured
    
    EXAMPLE:
    primary_storage = process_primary_storage(primary_storage, default_data['primary_storage'])
    """
    gdf = gdf.copy()
    
    # Phase 1: Fill from buffer
    fill_cols = ['LPG_capacity', 'LPG_price']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)
    
    # Phase 2: Ensure key fields and apply defaults
    if 'LPG_capacity' not in gdf.columns:
        gdf['LPG_capacity'] = 0.0
    else:
        gdf['LPG_capacity'] = gdf['LPG_capacity'].fillna(0.0)
    
    if 'LPG_price' not in gdf.columns:
        gdf['LPG_price'] = default_LPG_price
    else:
        gdf['LPG_price'] = gdf['LPG_price'].fillna(default_LPG_price)
    
    return gdf

def assign_nearest_tank_by_traveltime(facilities_gdf, storage_gdf, friction_path):
    """
    Assign nearest primary storage by travel time using OnStove friction routing.

    Adds three fields:
    - tank_name: nearest storage name by minimum travel time
    - tank_distance: straight-line distance in meters to selected storage
    - tank_traveltime: modeled travel time in hours to selected storage
    """
    facilities = facilities_gdf.copy()

    if facilities.empty:
        facilities['tank_name'] = pd.Series(dtype='object')
        facilities['tank_distance'] = pd.Series(dtype='float64')
        facilities['tank_traveltime'] = pd.Series(dtype='float64')
        return facilities

    if storage_gdf is None or storage_gdf.empty:
        facilities['tank_name'] = None
        facilities['tank_distance'] = np.nan
        facilities['tank_traveltime'] = np.nan
        return facilities

    friction_file = Path(friction_path)
    if not friction_file.exists():
        raise FileNotFoundError(f"Missing friction raster: {friction_file}")

    if facilities.crs is None:
        raise ValueError("Facilities layer has no CRS")
    if storage_gdf.crs is None:
        raise ValueError("primary_storage layer has no CRS")

    friction = RasterLayer(name='friction_surface', path=str(friction_file))
    friction_crs = friction.meta.get('crs')
    if friction_crs is None:
        raise ValueError("Friction raster has no CRS in metadata")

    facilities_tt = facilities.to_crs(friction_crs).copy()
    storage_tt = storage_gdf.to_crs(friction_crs).copy().reset_index(drop=True)

    name_col = 'name' if 'name' in storage_tt.columns else None
    if name_col is None:
        for col in storage_tt.columns:
            if str(col).lower() == 'name':
                name_col = col
                break
    if name_col is None:
        storage_tt['__tank_name'] = [f'primary_storage_{i}' for i in storage_tt.index]
        name_col = '__tank_name'
    else:
        storage_tt[name_col] = storage_tt[name_col].astype(str)
        missing_names = storage_tt[name_col].str.strip().isin(['', 'nan', 'None'])
        storage_tt.loc[missing_names, name_col] = [f'primary_storage_{i}' for i in storage_tt.index[missing_names]]

    facility_points = facilities_tt.geometry
    if not facilities_tt.geom_type.eq('Point').all():
        facility_points = facilities_tt.geometry.centroid

    storage_points = storage_tt.geometry
    if not storage_tt.geom_type.eq('Point').all():
        storage_points = storage_tt.geometry.centroid

    facility_rows = []
    facility_cols = []
    for geom in facility_points:
        row, col = rasterio.transform.rowcol(friction.meta['transform'], geom.x, geom.y)
        facility_rows.append(row)
        facility_cols.append(col)

    travel_matrix = np.full((len(facilities_tt), len(storage_tt)), np.nan, dtype=np.float64)

    for tank_idx, tank_geom in enumerate(storage_points):
        tank_layer = VectorLayer(name=f"tank_{tank_idx}")
        tank_layer.data = gpd.GeoDataFrame({'geometry': [tank_geom]}, geometry='geometry', crs=storage_tt.crs)

        tt_raster = tank_layer.travel_time(friction=friction, create_raster=False)
        tt_data = tt_raster.data
        max_row, max_col = tt_data.shape

        for fac_idx, (row, col) in enumerate(zip(facility_rows, facility_cols)):
            if 0 <= row < max_row and 0 <= col < max_col:
                val = tt_data[row, col]
                if np.isfinite(val):
                    travel_matrix[fac_idx, tank_idx] = float(val)

    tank_name = np.array([None] * len(facilities_tt), dtype=object)
    tank_distance = np.full(len(facilities_tt), np.nan, dtype=np.float64)
    tank_traveltime = np.full(len(facilities_tt), np.nan, dtype=np.float64)

    valid = np.any(np.isfinite(travel_matrix), axis=1)
    if np.any(valid):
        best_tank_idx = np.nanargmin(travel_matrix[valid], axis=1)
        valid_rows = np.where(valid)[0]

        storage_names = storage_tt[name_col].astype(str).to_numpy()
        tank_name[valid] = storage_names[best_tank_idx]
        tank_traveltime[valid] = travel_matrix[valid_rows, best_tank_idx]

        facilities_m = facilities_tt.to_crs('EPSG:3857')
        storage_m = storage_tt.to_crs('EPSG:3857')
        facility_points_m = facilities_m.geometry if facilities_m.geom_type.eq('Point').all() else facilities_m.geometry.centroid
        storage_points_m = storage_m.geometry if storage_m.geom_type.eq('Point').all() else storage_m.geometry.centroid

        for row_idx, tank_idx in zip(valid_rows, best_tank_idx):
            tank_distance[row_idx] = float(facility_points_m.iloc[row_idx].distance(storage_points_m.iloc[tank_idx]))

    facilities['tank_name'] = tank_name
    facilities['tank_distance'] = tank_distance
    facilities['tank_traveltime'] = tank_traveltime
    return facilities

# Apply processing for each facility type
refineries = process_refineries(refineries, default_data['refineries'])
primary_storage = process_primary_storage(primary_storage, default_data['primary_storage'])
ports = process_ports(ports, default_data['ports'], primary_storage, lpg_import_cost)
gas_plants = process_refineries(gas_plants, default_data['gas_plants'])  # Use same logic as refineries

# Assign nearest primary storage by travel time (OnStove)
friction_path = data_dir / 'friction_moto.tif'
refineries = assign_nearest_tank_by_traveltime(refineries, primary_storage, friction_path)
ports = assign_nearest_tank_by_traveltime(ports, primary_storage, friction_path)
gas_plants = assign_nearest_tank_by_traveltime(gas_plants, primary_storage, friction_path)

print("✓ Facilities processed in memory")
print("✓ tank_name, tank_distance and tank_traveltime added to all facility layers")


✓ Facilities processed in memory
✓ tank_name, tank_distance and tank_traveltime added to all facility layers


# calculate market share %

In [51]:
# STEP 4: Calculate percentages
def calculate_percentages(refineries_gdf, ports_gdf, gas_plants_gdf, nat_shares_df, storage_gdf):
    """
    CALCULATE FACILITY-LEVEL MARKET SHARE PERCENTAGES

    Computes market share by distributing national shares across individual facilities
    based on their production/storage capacity.

    OUTPUT:
    - result: combined GeoDataFrame for quick checks/summary
    - by_category: dict with one GeoDataFrame per source category, preserving original fields
    """

    # Calculate totals
    totals = {
        'refineries': refineries_gdf['LPG_prod'].sum(),
        'ports': ports_gdf['LPG_capacity'].sum(),
        'gas_plants': gas_plants_gdf['LPG_prod'].sum(),
    }

    # Get national shares from raster-derived values (no fallback defaults)
    nat_shares = {
        'refineries': nat_shares_df.loc['refineries'].iloc[0] / 100,
        'ports': nat_shares_df.loc['ports'].iloc[0] / 100,
        'gas_plants': nat_shares_df.loc['gas_plants'].iloc[0] / 100,
    }

    print(f"Totals: Refineries={totals['refineries']:.0f} | Ports={totals['ports']:.0f} | Gas={totals['gas_plants']:.0f}")
    print(f"Shares: {nat_shares}")

    by_category = {}
    specs = [
        ('refineries', refineries_gdf, 'LPG_prod'),
        ('ports', ports_gdf, 'LPG_capacity'),
        ('gas_plants', gas_plants_gdf, 'LPG_prod'),
    ]

    for src_name, src_gdf, prod_col in specs:
        gdf = src_gdf.copy()

        # Ensure required columns used later in the pipeline are present
        if 'name' not in gdf.columns:
            gdf['name'] = [f'{src_name}_{idx}' for idx in gdf.index]

        if 'LPG_price' not in gdf.columns:
            gdf['LPG_price'] = default_LPG_price
        else:
            gdf['LPG_price'] = gdf['LPG_price'].fillna(default_LPG_price)

        if 'country' not in gdf.columns:
            gdf['country'] = 'Unknown'
        else:
            gdf['country'] = gdf['country'].fillna('Unknown')

        # Keep category in each layer and compute percentage
        gdf['category'] = src_name if src_name != 'gas_plants' else 'gasplants'

        if prod_col not in gdf.columns:
            gdf[prod_col] = 0.0

        if totals[src_name] > 0:
            gdf['percentage'] = nat_shares[src_name] * (gdf[prod_col].fillna(0.0) / totals[src_name])
        else:
            gdf['percentage'] = 0.0

        by_category[src_name] = gdf

    result = gpd.GeoDataFrame(
        pd.concat([by_category['refineries'], by_category['ports'], by_category['gas_plants']], ignore_index=True),
        crs=refineries_gdf.crs,
    )

    print(f"✓ {len(result)} sources calculated | Total share: {result['percentage'].sum():.2%}")
    return result, by_category

# Use in-memory processed data
beginning, facilities_by_category = calculate_percentages(refineries, ports, gas_plants, national_shares, primary_storage)


Totals: Refineries=555219 | Ports=1204897 | Gas=470
Shares: {'refineries': np.float64(0.0006338856145062592), 'ports': np.float64(0.5073805527659039), 'gas_plants': np.float64(0.4919855616093956)}
✓ 18 sources calculated | Total share: 100.00%


# save output

In [59]:
# STEP 5: Save output as a multi-layer GPKG (facility layers + primary storage)
output_gpkg = data_dir / "beginning.gpkg"

layer_map = [
    ('refineries', 'refineries'),
    ('ports', 'ports'),
    ('gas_plants', 'gas_plants'),
    ('primary_storage', 'primary_storage'),
]

export_layers = {
    'refineries': facilities_by_category['refineries'],
    'ports': facilities_by_category['ports'],
    'gas_plants': facilities_by_category['gas_plants'],
    'primary_storage': primary_storage,
}

def _sanitize_for_gpkg(gdf):
    """Make a GeoDataFrame safe for GPKG writing while preserving all fields."""
    cleaned = gdf.copy()

    # Avoid case-insensitive duplicate field names in GPKG
    seen = {}
    new_cols = []
    for col in cleaned.columns:
        key = str(col).lower()
        if key in seen:
            seen[key] += 1
            new_cols.append(f"{col}_{seen[key]}")
        else:
            seen[key] = 0
            new_cols.append(col)
    cleaned.columns = new_cols

    # Convert problematic object fields to strings to avoid writer errors
    geom_col = cleaned.geometry.name if hasattr(cleaned, 'geometry') else 'geometry'
    for col in cleaned.columns:
        if col == geom_col:
            continue
        if cleaned[col].dtype == 'object':
            cleaned[col] = cleaned[col].apply(
                lambda v: v if isinstance(v, (str, int, float, bool, type(None), np.integer, np.floating)) else str(v)
            )

    return cleaned

if output_gpkg.exists():
    try:
        output_gpkg.unlink()
    except PermissionError as e:
        raise PermissionError(
            f"Cannot overwrite {output_gpkg}. Close any app using the file (QGIS/Explorer preview) and run this cell again."
        ) from e

try:
    for i, (key, layer_name) in enumerate(layer_map):
        gdf = _sanitize_for_gpkg(export_layers[key])
        mode = 'w' if i == 0 else 'a'
        gdf.to_file(output_gpkg, layer=layer_name, driver="GPKG", mode=mode)
except PermissionError as e:
    raise PermissionError(
        f"Cannot write to {output_gpkg}. Close any app using the file (QGIS/Explorer preview) and run this cell again."
    ) from e

# Keep combined dataframe available for summary and quick checks
output_gdf = beginning.copy()

print(f"\n✓ beginning.gpkg saved as multi-layer GPKG")
for key, layer_name in layer_map:
    gdf = export_layers[key]
    print(f"  - Layer '{layer_name}': {len(gdf)} records, {len(gdf.columns)} fields")
print(f"  Combined records (all layers): {len(output_gdf)}")



✓ beginning.gpkg saved as multi-layer GPKG
  - Layer 'refineries': 4 records, 14 fields
  - Layer 'ports': 11 records, 47 fields
  - Layer 'gas_plants': 3 records, 10 fields
  - Layer 'primary_storage': 19 records, 18 fields
  Combined records (all layers): 18


# print summary

In [60]:
# STEP 6: Summary
print("\n" + "="*70)
print("PROCESSING COMPLETE")
print("="*70)

print(f"\nConstants: default_LPG_price={default_LPG_price}, STS_cost={STS_cost}, BUFFER_KM={BUFFER_KM}")
print(f"\nSources by category:")
for cat, count in output_gdf.groupby('category').size().items():
    share = output_gdf[output_gdf['category'] == cat]['percentage'].sum()
    print(f"  - {cat}: {count} sources ({share:.1%} share)")

if 'country' in output_gdf.columns:
    print(f"\nSources by country:")
    for country in sorted(output_gdf['country'].dropna().unique()):
        records = output_gdf[output_gdf['country'] == country]
        print(f"  - {country}: {len(records)} sources ({records['percentage'].sum():.1%})")

print(f"\nOutputs: {data_dir}/")
if (data_dir / 'beginning.gpkg').exists():
    print("  ✓ beginning.gpkg")


PROCESSING COMPLETE

Constants: default_LPG_price=0.6, STS_cost=0.1, BUFFER_KM=10

Sources by category:
  - gasplants: 3 sources (49.2% share)
  - ports: 11 sources (50.7% share)
  - refineries: 4 sources (0.1% share)

Sources by country:
  - Nigeria: 14 sources (99.9%)
  - Unknown: 4 sources (0.1%)

Outputs: dataset_first_step/
  ✓ beginning.gpkg
